# GP fit against synthetic experimental correlated errors

This notebook builds a controlled synthetic version of the first `data/Table01.csv` centrality block. The bin positions and smooth central spectrum are taken from the data file, but the experimental errors are generated with a known RBF correlated component plus a known uncorrelated noise component.

The GP fit then receives only the synthetic central values at each point. It does not receive the true synthetic covariance matrix. The goal is to test whether a free-length RBF GP can recover a compatible correlation length and uncertainty structure from central values alone.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve, cholesky
from scipy.optimize import minimize

DATA_FILE = Path('data/Table01.csv')
OUT_DIR = Path('synthetic-rbf-output')
OUT_DIR.mkdir(exist_ok=True)

SEED = 20260904
TRUE_ELL = 0.3
TRUE_CORR_FRAC = 0.08
TRUE_NOISE_FRAC = 0.025
JITTER = 1e-10

rng = np.random.default_rng(SEED)


## Read the reference grid and central spectrum

The HEPData CSV contains several centrality blocks. We reuse the first block only to obtain realistic `pT` positions and a realistic positive spectrum shape.

In [ ]:
def read_first_centrality_block(path):
    rows = []
    centrality = None
    header = None
    reading_data = False

    with path.open('r', encoding='utf-8', newline='') as f:
        for row in csv.reader(f):
            if not row:
                if reading_data:
                    break
                continue

            first = row[0].strip()
            if first.startswith('#: CENTRALITY') and centrality is None:
                centrality = row[3].strip()
                continue

            if centrality is None:
                continue

            if first.startswith('$p_{T}$'):
                header = row
                reading_data = True
                continue

            if reading_data:
                rows.append([float(value) for value in row])

    if centrality is None or header is None or not rows:
        raise ValueError(f'Could not find the first centrality block in {path}')

    return centrality, np.asarray(rows, dtype=float)


centrality, raw = read_first_centrality_block(DATA_FILE)
pT = raw[:, 0]
pT_low = raw[:, 1]
pT_high = raw[:, 2]
y_reference = raw[:, 3]
log_y_true = np.log(y_reference)

print(f'Centrality block used as shape reference: {centrality}%')
print(f'Number of points: {len(pT)}')
print(f'pT range: {pT.min():.3g} to {pT.max():.3g} GeV/c')


## Synthetic correlated experimental errors

The synthetic observation is generated in log-space:

$$z_{obs} = z_{true} + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, K_{corr} + K_{noise}).$$

The correlated component is RBF with known length scale `TRUE_ELL = 0.3`. The diagonal noise is also known to the generator, but hidden from the GP fit.

In [ ]:
def rbf_correlation(x1, x2, ell):
    x1 = np.asarray(x1, dtype=float).reshape(-1, 1)
    x2 = np.asarray(x2, dtype=float).reshape(1, -1)
    return np.exp(-0.5 * ((x1 - x2) / ell) ** 2)


def stable_cholesky(K, jitter=JITTER):
    jitter_local = jitter
    for _ in range(10):
        try:
            return cholesky(K + jitter_local * np.eye(K.shape[0]), lower=True, check_finite=False)
        except np.linalg.LinAlgError:
            jitter_local *= 10.0
    raise np.linalg.LinAlgError('Cholesky failed even after jitter escalation.')


R_true = rbf_correlation(pT, pT, TRUE_ELL)
K_corr_true_log = TRUE_CORR_FRAC**2 * R_true
K_noise_true_log = TRUE_NOISE_FRAC**2 * np.eye(len(pT))
K_total_true_log = K_corr_true_log + K_noise_true_log

L_true = stable_cholesky(K_total_true_log)
eps_log = L_true @ rng.normal(size=len(pT))
log_y_obs = log_y_true + eps_log
y_obs = np.exp(log_y_obs)

sigma_corr_true_abs = y_reference * TRUE_CORR_FRAC
sigma_noise_true_abs = y_reference * TRUE_NOISE_FRAC

synthetic_data = pd.DataFrame({
    'pT': pT,
    'pT_low': pT_low,
    'pT_high': pT_high,
    'y_true': y_reference,
    'y_synthetic': y_obs,
    'log_error_draw': eps_log,
    'sigma_corr_true_abs': sigma_corr_true_abs,
    'sigma_noise_true_abs': sigma_noise_true_abs,
})
synthetic_data.to_csv(OUT_DIR / 'synthetic_rbf_dataset.csv', index=False)
synthetic_data.head(12)


## Fit an RBF GP using central values only

The GP is fitted to the centered log central values. The learned parameters are `sigma_f`, `ell`, and `sigma_n`.

In [ ]:
z = log_y_obs - np.mean(log_y_obs)


def negative_log_marginal_likelihood(theta):
    log_sigma_f, log_ell, log_sigma_n = theta
    sigma_f = np.exp(log_sigma_f)
    ell = np.exp(log_ell)
    sigma_n = np.exp(log_sigma_n)

    R = rbf_correlation(pT, pT, ell)
    K = sigma_f**2 * R + sigma_n**2 * np.eye(len(pT)) + JITTER * np.eye(len(pT))
    K = 0.5 * (K + K.T)

    try:
        c, lower = cho_factor(K, lower=True, check_finite=False)
        alpha = cho_solve((c, lower), z, check_finite=False)
        logdet = 2.0 * np.sum(np.log(np.diag(c)))
        return 0.5 * z.dot(alpha) + 0.5 * logdet + 0.5 * len(pT) * np.log(2.0 * np.pi)
    except np.linalg.LinAlgError:
        return 1e50


def fit_gp_free_ell():
    z_std = max(float(np.std(z)), 1e-6)
    starts = [
        [z_std, 0.2, 0.05 * z_std],
        [z_std, 0.3, 0.10 * z_std],
        [2.0 * z_std, 0.5, 0.10 * z_std],
        [0.5 * z_std, 1.0, 0.20 * z_std],
    ]
    bounds = [
        (np.log(1e-5), np.log(10.0)),
        (np.log(0.03), np.log(10.0)),
        (np.log(1e-6), np.log(5.0)),
    ]

    results = []
    for start in starts:
        res = minimize(
            negative_log_marginal_likelihood,
            np.log(start),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 10000, 'ftol': 1e-12, 'gtol': 1e-8},
        )
        if np.isfinite(res.fun):
            results.append(res)

    best = min(results, key=lambda item: item.fun)
    sigma_f, ell, sigma_n = np.exp(best.x)
    return {
        'sigma_f': float(sigma_f),
        'ell': float(ell),
        'sigma_n': float(sigma_n),
        'nll': float(best.fun),
        'success': bool(best.success),
        'message': str(best.message),
    }


fit = fit_gp_free_ell()
fit


In [ ]:
fit_table = pd.DataFrame([
    {'quantity': 'true RBF ell', 'value': TRUE_ELL},
    {'quantity': 'fitted RBF ell', 'value': fit['ell']},
    {'quantity': 'true correlated fractional amplitude', 'value': TRUE_CORR_FRAC},
    {'quantity': 'fitted GP sigma_f on log(y)', 'value': fit['sigma_f']},
    {'quantity': 'true uncorrelated fractional noise', 'value': TRUE_NOISE_FRAC},
    {'quantity': 'fitted GP sigma_n on log(y)', 'value': fit['sigma_n']},
    {'quantity': 'negative log marginal likelihood', 'value': fit['nll']},
])
fit_table.to_csv(OUT_DIR / 'synthetic_rbf_fit_hyperparameters.csv', index=False)
fit_table


## Posterior prediction and uncertainty decomposition

In [ ]:
def gp_predict_log(x_train, z_train, x_test, sigma_f, ell, sigma_n):
    x_train = np.asarray(x_train, dtype=float)
    x_test = np.asarray(x_test, dtype=float)
    R = rbf_correlation(x_train, x_train, ell)
    K = sigma_f**2 * R + sigma_n**2 * np.eye(len(x_train)) + JITTER * np.eye(len(x_train))
    K = 0.5 * (K + K.T)
    K_s = sigma_f**2 * rbf_correlation(x_test, x_train, ell)
    K_ss_diag = np.full(len(x_test), sigma_f**2)

    c, lower = cho_factor(K, lower=True, check_finite=False)
    alpha = cho_solve((c, lower), z_train, check_finite=False)
    mu_centered = K_s @ alpha
    v = cho_solve((c, lower), K_s.T, check_finite=False)
    var_latent = np.maximum(K_ss_diag - np.sum(K_s * v.T, axis=1), 0.0)
    return mu_centered + np.mean(log_y_obs), var_latent


x_grid = np.linspace(pT.min(), pT.max(), 600)
log_mu_grid, log_var_corr_grid = gp_predict_log(pT, z, x_grid, **{k: fit[k] for k in ['sigma_f', 'ell', 'sigma_n']})
log_mu_train, log_var_corr_train = gp_predict_log(pT, z, pT, **{k: fit[k] for k in ['sigma_f', 'ell', 'sigma_n']})

gp_mean_grid = np.exp(log_mu_grid)
gp_mean_train = np.exp(log_mu_train)
sigma_corr_gp_grid = gp_mean_grid * np.sqrt(log_var_corr_grid)
sigma_corr_gp_train = gp_mean_train * np.sqrt(log_var_corr_train)
sigma_noise_gp_train = gp_mean_train * fit['sigma_n']
sigma_noise_gp_grid = gp_mean_grid * fit['sigma_n']

true_corr_grid = np.exp(np.interp(x_grid, pT, log_y_true)) * TRUE_CORR_FRAC
true_noise_grid = np.exp(np.interp(x_grid, pT, log_y_true)) * TRUE_NOISE_FRAC


## Pointwise ratios and table

Ratios equal to one indicate exact agreement with the synthetic truth used by the generator.

In [ ]:
def safe_ratio(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    return np.divide(num, den, out=np.full_like(num, np.nan, dtype=float), where=den != 0)


comparison = pd.DataFrame({
    'pT': pT,
    'y_true': y_reference,
    'y_synthetic': y_obs,
    'gp_mean': gp_mean_train,
    'ratio_gp_mean_over_synthetic': safe_ratio(gp_mean_train, y_obs),
    'true_uncorrelated_sigma': sigma_noise_true_abs,
    'gp_uncorrelated_sigma': sigma_noise_gp_train,
    'ratio_gp_uncorr_over_true': safe_ratio(sigma_noise_gp_train, sigma_noise_true_abs),
    'true_correlated_sigma': sigma_corr_true_abs,
    'gp_correlated_sigma': sigma_corr_gp_train,
    'ratio_gp_corr_over_true': safe_ratio(sigma_corr_gp_train, sigma_corr_true_abs),
})
comparison.to_csv(OUT_DIR / 'synthetic_rbf_pointwise_ratios.csv', index=False)
comparison.head(15)


In [ ]:
ratio_summary = comparison[[
    'ratio_gp_mean_over_synthetic',
    'ratio_gp_uncorr_over_true',
    'ratio_gp_corr_over_true',
]].agg(['mean', 'median', 'std', lambda s: np.sqrt(np.nanmean((s - 1.0)**2))]).T
ratio_summary = ratio_summary.rename(columns={'<lambda>': 'rms_ratio_minus_one'})
ratio_summary.to_csv(OUT_DIR / 'synthetic_rbf_ratio_summary.csv')
ratio_summary


## Plots

In [ ]:
plt.style.use('default')

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.set_yscale('log')
ax.plot(x_grid, np.exp(np.interp(x_grid, pT, log_y_true)), linestyle='--', label='synthetic truth')
ax.scatter(pT, y_obs, s=22, label='synthetic central values')
ax.plot(x_grid, gp_mean_grid, label='GP posterior mean')
ax.fill_between(x_grid, gp_mean_grid - 2.0 * sigma_corr_gp_grid, gp_mean_grid + 2.0 * sigma_corr_gp_grid, alpha=0.25, label='GP correlated band')
ax.fill_between(x_grid, np.exp(np.interp(x_grid, pT, log_y_true)) - 2.0 * true_corr_grid, np.exp(np.interp(x_grid, pT, log_y_true)) + 2.0 * true_corr_grid, alpha=0.18, label='true correlated band')
ax.set_xlabel(r'$p_T$ [GeV/c]')
ax.set_ylabel(r'$(1/N_{ev}) d^2N/(dp_T dy)$')
ax.set_title('Synthetic RBF correlated-error data and GP fit')
ax.legend(frameon=False)
fig.savefig(OUT_DIR / 'synthetic_rbf_gp_fit.png', dpi=200)
fig.savefig(OUT_DIR / 'synthetic_rbf_gp_fit.pdf')
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, constrained_layout=True)
ratio_columns = [
    ('ratio_gp_mean_over_synthetic', 'GP mean / synthetic central value'),
    ('ratio_gp_uncorr_over_true', 'GP uncorrelated sigma / true noise sigma'),
    ('ratio_gp_corr_over_true', 'GP correlated sigma / true correlated sigma'),
]
for ax, (col, label) in zip(axes, ratio_columns):
    ax.plot(comparison['pT'], comparison[col], marker='o')
    ax.axhline(1.0, color='0.35', linestyle='--', linewidth=1.0)
    ax.set_ylabel('ratio')
    ax.set_title(label)
axes[-1].set_xlabel(r'$p_T$ [GeV/c]')
fig.savefig(OUT_DIR / 'synthetic_rbf_pointwise_ratios.png', dpi=200)
fig.savefig(OUT_DIR / 'synthetic_rbf_pointwise_ratios.pdf')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
im0 = axes[0].imshow(R_true, vmin=0.0, vmax=1.0, origin='lower')
axes[0].set_title(f'True RBF correlation, ell = {TRUE_ELL}')
axes[0].set_xlabel('point j')
axes[0].set_ylabel('point i')
fig.colorbar(im0, ax=axes[0])

R_fit = rbf_correlation(pT, pT, fit['ell'])
im1 = axes[1].imshow(R_fit, vmin=0.0, vmax=1.0, origin='lower')
axes[1].set_title(f'Fitted RBF correlation, ell = {fit["ell"]:.3g}')
axes[1].set_xlabel('point j')
axes[1].set_ylabel('point i')
fig.colorbar(im1, ax=axes[1])

fig.savefig(OUT_DIR / 'synthetic_rbf_correlation_matrices.png', dpi=200)
fig.savefig(OUT_DIR / 'synthetic_rbf_correlation_matrices.pdf')
plt.show()


## Interpretation checklist

Important caveat: with a single synthetic central-value curve, the optimized GP length scale is mostly informed by the smoothness of the observed spectrum. It is not guaranteed to recover the RBF correlation length used to generate the experimental error field. Recovering the error covariance itself requires either repeated pseudo-experiments, explicit covariance likelihood modeling, or additional information about the experimental uncertainty decomposition.

- Does the optimized `ell` approach the known synthetic value `0.3`?
- Are the pointwise uncertainty ratios close to one, or does the GP redistribute correlated and uncorrelated variance?
- Does fitting only central values provide enough information to identify the imposed experimental covariance, or only a compatible smoothness scale?